# Alzheimer's OASIS-1 — ANFIS 评估

读取 `alzheimer_anfis_train_colab.ipynb` 产出的 out-of-fold 预测（每个受试者恰好被
评估一次，最大化有效评估样本量），计算 accuracy/sensitivity/specificity/AUC-ROC、
混淆矩阵，`ANFIS-on-GT` 和 `ANFIS-on-extracted-hippo` 两个配置分别报告——
两者差距大说明瓶颈在 HippoDeep 海马体积提取质量，差距小说明 ANFIS 本身已经接近特征质量的上限。

最后对几个已知 CDR 的受试者生成规则触发解释文本，人工核对是否合理。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'scikit-learn', 'matplotlib'])

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, classification_report

DRIVE_ROOT = '/content/drive/MyDrive'
OUTPUT_DIR = f'{DRIVE_ROOT}/alzheimer_anfis_outputs'
RESULTS_DIR = f'{OUTPUT_DIR}/results'

CLASS_ORDER = ['Nondemented', 'Demented']
SEVERITY_THRESHOLD = 0.25   # CDR>0 分界，和 ANFIS 训练/规则表里用的阈值保持一致

In [ ]:
# ── anfis_score_to_probs 适配器：把连续严重度分数转成类似概率的形式 ──
# 单调变换不影响 AUC-ROC 排序，只是为了让阈值化的判断更直观、也方便画 ROC 曲线。

def anfis_score_to_probs(severity_scores: np.ndarray, temperature: float = 0.5) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-(severity_scores - SEVERITY_THRESHOLD) / temperature))


def compute_metrics(labels_binary: np.ndarray, probs_demented: np.ndarray, threshold: float = 0.5) -> dict:
    preds = (probs_demented >= threshold).astype(int)
    auc = roc_auc_score(labels_binary, probs_demented)
    cm = confusion_matrix(labels_binary, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        'n': len(labels_binary),
        'accuracy': (tp + tn) / (tp + tn + fp + fn),
        'auc_roc': auc,
        'sensitivity': tp / (tp + fn + 1e-9),
        'specificity': tn / (tn + fp + 1e-9),
        'cm': cm,
        'report': classification_report(labels_binary, preds, target_names=CLASS_ORDER, digits=4),
        'labels': labels_binary,
        'probs': probs_demented,
    }


def plot_confusion_matrix(cm: np.ndarray, title: str, save_path: str) -> None:
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(CLASS_ORDER, rotation=20, ha='right')
    ax.set_yticklabels(CLASS_ORDER)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    th = cm.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > th else 'black', fontsize=13)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()


def plot_roc(results: dict, save_path: str) -> None:
    fig, ax = plt.subplots(figsize=(6, 6))
    for name, r in results.items():
        fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
        ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={r["auc_roc"]:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve — ANFIS (out-of-fold)')
    ax.legend(loc='lower right')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()


print('评估函数定义完成。')

In [ ]:
# ── 读取两个配置的 out-of-fold 预测，计算指标 ─────────────────────
config_files = {
    'ANFIS-on-GT': f'{RESULTS_DIR}/oof_predictions_ANFIS-on-GT.csv',
    'ANFIS-on-extracted-hippo': f'{RESULTS_DIR}/oof_predictions_ANFIS-on-extracted-hippo.csv',
}

eval_results = {}
for config_name, path in config_files.items():
    if not os.path.exists(path):
        print(f'找不到 {path}，跳过（去 anfis_train notebook 里确认对应配置是否训练成功）')
        continue
    df = pd.read_csv(path)
    labels_binary = (df['label'] == 'Demented').astype(int).to_numpy()
    probs = anfis_score_to_probs(df['oof_pred_severity'].to_numpy())
    metrics = compute_metrics(labels_binary, probs, threshold=0.5)
    eval_results[config_name] = metrics

    print(f'\n=== {config_name} (out-of-fold, N={metrics["n"]}) ===')
    print(f'Accuracy    : {metrics["accuracy"]:.4f}')
    print(f'AUC-ROC     : {metrics["auc_roc"]:.4f}')
    print(f'Sensitivity : {metrics["sensitivity"]:.4f}')
    print(f'Specificity : {metrics["specificity"]:.4f}')
    print(metrics['report'])

    cm_path = f'{RESULTS_DIR}/confusion_matrix_{config_name.replace(" ", "_")}.png'
    plot_confusion_matrix(metrics['cm'], f'Confusion Matrix — {config_name}', cm_path)
    print(f'混淆矩阵图: {cm_path}')

if len(eval_results) == 2:
    roc_path = f'{RESULTS_DIR}/roc_comparison_gt_vs_extracted.png'
    plot_roc(eval_results, roc_path)
    print(f'\nROC 对比图: {roc_path}')
    gap = eval_results['ANFIS-on-GT']['auc_roc'] - eval_results['ANFIS-on-extracted-hippo']['auc_roc']
    print(f'\nGT 配置与提取特征配置的 AUC 差距: {gap:.4f}')
    print('差距大 -> 瓶颈在 HippoDeep 海马体积提取质量；差距小 -> ANFIS 本身已经接近特征质量的上限。')

In [ ]:
# ── 阈值调优（Youden's J）─────────────────────────────────────────
# 上面固定用 0.5 阈值（对应 severity=0.25），不一定是当前模型最优的分类切点。
#
# 直接在 out-of-fold 预测上找 Youden's J 最优阈值、再拿同一批 OOF 预测报指标，
# 属于用同一份"测试集"既调参又汇报结果，会有轻微的过于乐观偏差（不是选到了真正
# 分布外的数据，但阈值毕竟是照着这批数据的标签挑出来的）。
# 做法：把 OOF 数据随机切成两半，一半专门用来找阈值，另一半专门用来报最终指标，
# 重复多次不同的随机切分取平均，既避免用同一份数据既调阈值又汇报，又不会因为
# 只看一次切分而对某个偶然的切分结果过拟合。

from sklearn.metrics import f1_score as _f1_score
from sklearn.model_selection import train_test_split

N_THRESHOLD_REPEATS = 20


def find_optimal_threshold(labels: np.ndarray, probs: np.ndarray, metric: str = 'youden') -> float:
    fpr, tpr, thresholds = roc_curve(labels, probs)
    if metric == 'youden':
        best_idx = int(np.argmax(tpr - fpr))
    else:
        f1s = [_f1_score(labels, (probs >= t).astype(int), zero_division=0) for t in thresholds]
        best_idx = int(np.argmax(f1s))
    return float(thresholds[best_idx])


def evaluate_with_tuned_threshold(df: pd.DataFrame, n_repeats: int = N_THRESHOLD_REPEATS) -> dict:
    labels_binary = (df['label'] == 'Demented').astype(int).to_numpy()
    probs_all = anfis_score_to_probs(df['oof_pred_severity'].to_numpy())

    thresholds, accs, senss, specs = [], [], [], []
    for seed in range(n_repeats):
        idx_tune, idx_report = train_test_split(
            np.arange(len(df)), test_size=0.5, stratify=labels_binary, random_state=seed)
        t = find_optimal_threshold(labels_binary[idx_tune], probs_all[idx_tune], metric='youden')
        m = compute_metrics(labels_binary[idx_report], probs_all[idx_report], threshold=t)
        thresholds.append(t)
        accs.append(m['accuracy'])
        senss.append(m['sensitivity'])
        specs.append(m['specificity'])

    return {
        'threshold_mean': np.mean(thresholds), 'threshold_std': np.std(thresholds),
        'accuracy_mean': np.mean(accs), 'accuracy_std': np.std(accs),
        'sensitivity_mean': np.mean(senss), 'sensitivity_std': np.std(senss),
        'specificity_mean': np.mean(specs), 'specificity_std': np.std(specs),
    }


for config_name, path in config_files.items():
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    r = evaluate_with_tuned_threshold(df)
    print(f'\n=== {config_name}: 阈值调优后（{N_THRESHOLD_REPEATS} 次随机切分取均值±标准差）===')
    print(f'最优阈值    : {r["threshold_mean"]:.3f} ± {r["threshold_std"]:.3f}  (默认阈值是 0.5)')
    print(f'Accuracy    : {r["accuracy_mean"]:.4f} ± {r["accuracy_std"]:.4f}  '
          f'(固定0.5阈值下是 {eval_results[config_name]["accuracy"]:.4f})')
    print(f'Sensitivity : {r["sensitivity_mean"]:.4f} ± {r["sensitivity_std"]:.4f}  '
          f'(固定0.5阈值下是 {eval_results[config_name]["sensitivity"]:.4f})')
    print(f'Specificity : {r["specificity_mean"]:.4f} ± {r["specificity_std"]:.4f}  '
          f'(固定0.5阈值下是 {eval_results[config_name]["specificity"]:.4f})')
    print('（AUC 是阈值无关的，不会因为调阈值而变化，上面已经报过了）')

In [ ]:
# ── 对几个受试者生成规则触发解释文本（人工核对用）────────────────
# 这里直接从 out-of-fold 预测表里挑样本；如果需要看"哪些规则触发"的完整细节
# （而不只是最终分数），要回到 anfis_train notebook 里，用对应折的 checkpoint
# 重新跑一次 explain_prediction()（这里只是基于已保存的 CSV 做摘要展示）。

for config_name, path in config_files.items():
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    print(f'\n=== {config_name}: 抽样核对 ===')
    sample = df.sample(min(5, len(df)), random_state=42)
    for _, row in sample.iterrows():
        print(f"{row['subject_id']}: 真实CDR={row['cdr_numeric']:.1f} ({row['label']}) | "
              f"ANFIS预测severity={row['oof_pred_severity']:.2f} -> {row['oof_pred_label']} | "
              f"{'✅ 一致' if row['label'] == row['oof_pred_label'] else '❌ 不一致'}")